In [34]:
import pandas as pd
from nltk.corpus import stopwords
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import numpy as np

In [35]:
stop_words = set(stopwords.words('english'))

In [36]:
#Nettoyage des données
def text_process(mess):
    lower_mess = mess.lower()
    nopunc = [char for char in lower_mess if char not in string.punctuation]
    nopunc = ''.join(nopunc)
    clean_mess = [word for word in nopunc.split() if word not in stop_words]
    return clean_mess

In [37]:
data = pd.read_csv('../../data/train_with_sentiments.csv')
data = data.dropna()
data = data.drop_duplicates(subset=['Context','Response'])
data = data.drop_duplicates(subset=['Context'])
data = data.drop_duplicates(subset=['Response'])
test = data.sample(n=10, random_state=42)
data = data.drop(test.index)
data.to_csv('../../data/train_unique_e2p2.csv', index=False)
test.to_csv('../../data/test.csv', index=False)

In [38]:
train_data = pd.read_csv('../../data/train_unique_e2p2.csv')
test_data = pd.read_csv('../../data/test.csv')

In [39]:
train_data["Context_clean"] = train_data["Context"].apply(text_process)
train_data["Response_clean"] = train_data["Response"].apply(text_process)
# Structure du dictionnaire : mot -> liste des discussions contenant ce type de mot
index_inverse = {}
# Nombre minimum d'occurrences pour garder une discussion
SEUIL = 1
for idx, row in train_data.iterrows():
    question = row["Context"]
    response = row["Response"]
    #combine les mots de la question et de la réponse
    mots = row["Context_clean"] + row["Response_clean"]
    # Compter le nombre d'apparitions de chaque mot
    from collections import Counter
    compteur = Counter(mots)
    for mot,count in compteur.items():
        if count >= SEUIL:
        #Si le mot n'existe pas encore dans le dictionnaire, on l'initialise
            if mot not in index_inverse:
                index_inverse[mot] = []
            #On ajoute la discussion associée à ce mot dans la liste
            index_inverse[mot].append({
                "id": idx,
                "question": question,
                "response": response
            })

In [40]:
anxiete = index_inverse.get("anxiety", [])
print("Nombre de discussion contenant le mot anxiety plus de",SEUIL,"fois :", len(anxiete))
print("\nDiscussion pour le mot 'anxiety' :\n")
#Pour un affichage plus propre
for i, item in enumerate(anxiete, 1):
    print(f"--- Exemple {i} ---")
    print(f"ID       : {item['id']}")
    print(f"Question : {item['question']}")
    print(f"Réponse  : {item['response']}\n")

Nombre de discussion contenant le mot anxiety plus de 1 fois : 116

Discussion pour le mot 'anxiety' :

--- Exemple 1 ---
ID       : 1
Question : I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.
   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?
Réponse  : Let me start by saying there are never too many concerns that you can bring into counselling. In fact, most people who come to see me for counselling have more than one issue they would like to work on in psychotherapy and most times these are all interconnected. In counselling, we work together, collaboratively, to figure out which issues you would like to address first and then together we develop an individualized plan of care. Basically, it’s like a road map 

In [43]:
# Méthode 1 : TF-IDF
vectorizer = TfidfVectorizer(analyzer=text_process)
#X_train = vectorizer.fit_transform(train_data['Context'])
def questionQuestion(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_bert'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_bert'] == sentiment]
    X_train = vectorizer.fit_transform(data_sentiment['Context'])
    X_question = vectorizer.transform([question])
    similarities = cosine_similarity(X_question, X_train).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    list = []
    for idx in top_indices:
        list.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return list

In [44]:
# Test méthode 1 : TF-IDF
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:
- (Cosine Similarity: 0.2463) Fears are not that difficult to deal with, first you need to train yourself to relax using some relaxation strategy, once you are able to employ that in your daily life, you then need to start facing your fear, for instance I'll use an e

In [45]:
# Méthode 1 : Word2Vec
phrases_train = train_data["Context"].apply(text_process).tolist()
model_w2v = Word2Vec(sentences=phrases_train, vector_size=100, window=5, min_count=1, workers=4)
def vectoriser_phrase(phrase):
    mots = text_process(phrase)
    vecteurs = [model_w2v.wv[mot] for mot in mots if mot in model_w2v.wv]
    if vecteurs:
        return np.mean(vecteurs, axis=0)
    else:
        return np.zeros(model_w2v.vector_size)
# X_train_w2v = np.array([vectoriser_phrase(phrase) for phrase in train_data["Context"]])
def questionQuestion_w2v(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_bert'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_bert'] == sentiment]
    X_train_w2v = np.array([vectoriser_phrase(phrase) for phrase in data_sentiment["Context"]])
    vecteur_question = vectoriser_phrase(question).reshape(1, -1)
    similarities = cosine_similarity(vecteur_question, X_train_w2v).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    list = []
    for idx in top_indices:
        list.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return list


In [46]:
# Test méthode 1 : Word2Vec
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion_w2v(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:
- (Cosine Similarity: 1.0000) It's never to late to get help with grief.  Get help as soon as possible before you are feeling the same way 5 years from now.  You will always miss your Dad but getting help with coping with his loss will make life easier to live.
- (Co

In [47]:
# Méthode 1 : Bert
model = SentenceTransformer('all-MiniLM-L6-v2')
#question_bert = model.encode(train_data["Context"].tolist())
def questionQuestion_bert(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_bert'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_bert'] == sentiment]
    question_bert = model.encode(data_sentiment["Context"].tolist())
    question_embedding = model.encode([question])
    cosine_similarities = cosine_similarity(question_embedding,question_bert).flatten()
    top_k_indices = cosine_similarities.argsort()[::-1][:k]
    list = []
    for idx in top_k_indices:
        list.append((train_data.iloc[idx]["Response"], cosine_similarities[idx]))
    return list

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [48]:
# Test méthode 1 : Bert
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion_bert(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:
- (Cosine Similarity: 0.4970) Anxiety is usually a sign of a current problem to which familiar emotional patterns of feeling similarly upset, attach themselves.Try to understand more about who you are, what you like, feel uneasy about, especially your deeper emotions

In [49]:
# Méthde 2 : TF-IDF 
vectorizer = TfidfVectorizer(analyzer=text_process)
# X_train = vectorizer.fit_transform(train_data['Response'])
def questionQuestion2(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_bert'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_bert'] == sentiment]
    X_train = vectorizer.fit_transform(data_sentiment['Response'])
    question_vector = vectorizer.transform([question])
    similarities = cosine_similarity(question_vector, X_train).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    list = []
    for idx in top_indices:
        list.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return list

In [50]:
# Test méthode 2 : TF-IDF
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion2(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:
- (Cosine Similarity: 0.1779) There's a quote I love that says, "Wherever you go, there you are" and the book by the same name by Jon Kabat-Zinn may be very helpful for you. The thing about changing things up when they get tough is that they often aren't the things t

In [51]:
# Méthode 2 : Word2Vec
phrases_train = train_data["Response"].apply(text_process).tolist()
model_w2v = Word2Vec(sentences=phrases_train, vector_size=100, window=5, min_count=2)
def vectoriser_phrase2(phrase):
    mots = text_process(phrase)
    vecteurs = [model_w2v.wv[mot] for mot in mots if mot in model_w2v.wv]
    if vecteurs:
        return np.mean(vecteurs, axis=0)
    else:
        return np.zeros(model_w2v.vector_size)
# X_train_w2v = np.array([vectoriser_phrase2(phrase) for phrase in train_data["Response"]])
def questionQuestion2_w2v(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_bert'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_bert'] == sentiment]
    X_train_w2v = np.array([vectoriser_phrase2(phrase) for phrase in data_sentiment["Response"]])
    vecteur_question = vectoriser_phrase2(question).reshape(1, -1)
    similarities = cosine_similarity(vecteur_question, X_train_w2v).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    results = []
    for idx in top_indices:
        results.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return results


In [52]:
# Test méthode 2 : Word2Vec
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion2_w2v(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:
- (Cosine Similarity: 1.0000) Are there any times or moments in which you feel other than "empty"?  Start with knowing the context of when you feel something other than empty.If there is no recent example, then in your mind, go back in time to think of when you felt 

In [53]:
# Méthode 2 : Bert
model = SentenceTransformer('all-MiniLM-L6-v2')
# question_bert = model.encode(train_data["Response"].tolist())
def questionQuestion2_bert(question,k=5):
    sentiment = test_data.loc[test_data['Context'] == question, 'predicted_sentiment_bert'].values[0]
    data_sentiment = train_data[train_data['predicted_sentiment_bert'] == sentiment]
    question_bert = model.encode(data_sentiment["Response"].tolist())
    question_embedding = model.encode([question])
    cosine_similarities = cosine_similarity(question_embedding,question_bert).flatten()
    top_k_indices = cosine_similarities.argsort()[::-1][:k]
    res = []
    for idx in top_k_indices:
        res.append((train_data.iloc[idx]["Response"], cosine_similarities[idx]))
    return res

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [54]:
# Test méthode 2 : Bert
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion2_bert(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: He was in love with someone years ago, and he still thinks about her time to time. He said, and I quote, "That relationship is definitely over. I love you, but that girl will always be in my mind." It just didn't feel like he appreciated all the things I've done to make him happy.
Actual Response: Trust your intuition on your conclusion about this guy.He may very well love you, only with the ex so prominent in his mind, it is possible your feeling of not being appreciated now, would multiply if ever the two of you needed to address a delicate topic.Since he is emotionally attached to the former gf, it is very likely he wouldn't be able to fully love you as much as you'd like and are already sensing.
Top response:
- (Cosine Similarity: 0.4240) Anxiety is usually a sign of a current problem to which familiar emotional patterns of feeling similarly upset, attach themselves.Try to understand more about who you are, what you like, feel uneasy about, especially your deeper emotions

In [55]:
X = model.encode(test_data['Response'].tolist())
def eval_mrr_bert(predictions):
    mrr_total = 0
    for i in range(len(test_data)):
        actualResponse = X[i].reshape(1, -1)
        predictedResponse = model.encode([predictions[i]])[0].reshape(1, -1)
        similarity = cosine_similarity(predictedResponse, actualResponse)[0]
        score = similarity.max()
        print(score)
        if score > 0:
            mrr_total += score
    return mrr_total / len(test_data)
predictions = []
for question in test_data['Context']:
    predicted_response = questionQuestion2_bert(question)[0][0]
    predictions.append(predicted_response)
res = eval_mrr_bert(predictions)
print(f"MRR: {res:.4f}")

0.15442526
0.10436077
0.45342237
0.030117216
0.12245801
0.3249693
0.39630872
0.3370431
0.024302186
0.3525417
MRR: 0.2300
